# Customising UC2 Red Agents

© Crown-owned copyright 2025, Defence Science and Technology Laboratory UK

This notebook will go over some examples of how UC2 red agent behaviour can be varied by changing its configuration parameters.

First, let's load the standard Data Manipulation config file, and see what the red agent does.

*For a full explanation of the Data Manipulation scenario (also known as UC2), check out the [data manipulation scenario notebook](./Data-Manipulation-E2E-Demonstration.ipynb)*

In [1]:
!primaite setup

2025-03-21 11:07:46,188: Performing the PrimAITE first-time setup...
2025-03-21 11:07:46,188: Building the PrimAITE app directories...
2025-03-21 11:07:46,188: Building primaite_config.yaml...
2025-03-21 11:07:46,188: Rebuilding the demo notebooks...
2025-03-21 11:07:46,212: Rebuilding the example notebooks...
2025-03-21 11:07:46,214: PrimAITE setup complete!


In [2]:
# Imports

from primaite.config.load import data_manipulation_config_path
from primaite.game.agent.interface import AgentHistoryItem
from primaite.session.environment import PrimaiteGymEnv
import yaml
from pprint import pprint

In [3]:
def make_cfg_have_flat_obs(cfg):
    for agent in cfg['agents']:
        if agent['type'] == "proxy-agent":
            agent['agent_settings']['flatten_obs'] = False

In [4]:
with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)
    make_cfg_have_flat_obs(cfg)

env = PrimaiteGymEnv(env_config = cfg)
obs, info = env.reset()
print('env created successfully')

2025-03-21 11:07:49,864: PrimaiteGymEnv RNG seed = None


2025-03-21 11:07:49,865: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:07:49,866: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


env created successfully


In [5]:
def friendly_output_red_action(info):
    # parse the info dict form step output and write out what the red agent is doing
    red_info : AgentHistoryItem = info['agent_actions']['data_manipulation_attacker']
    red_action = red_info.action
    if red_action == 'do-nothing':
        red_str = 'DO NOTHING'
    elif red_action == 'node-application-execute':
        red_str = f"ATTACK from {red_info.parameters['node_name']}"
    return red_str

By default, the red agent can start on client 1 or client 2. It starts its attack on a random step between 20 and 30, and it repeats its attack every 15-25 steps.

It also has a 20% chance to fail to perform the port scan, and a 20% chance to fail launching the SQL attack. However it will continue where it left off after a failed step. I.e. if lucky, it can perform the port scan and SQL attack on the first try. If the port scan works, but the sql attack fails the first time it tries to attack, the next time it will not need to port scan again, it can go straight to trying to use SQL attack again.

In [6]:
for step in range(35):
    step_num = env.game.step_counter
    obs, reward, terminated, truncated, info = env.step(0)
    red = friendly_output_red_action(info)
    print(f"step: {step_num:3}, Red action: {friendly_output_red_action(info)}, Blue reward:{reward:.2f}" )

step:   0, Red action: DO NOTHING, Blue reward:0.40
step:   1, Red action: DO NOTHING, Blue reward:0.90
step:   2, Red action: DO NOTHING, Blue reward:0.90
step:   3, Red action: DO NOTHING, Blue reward:0.95
step:   4, Red action: DO NOTHING, Blue reward:0.95
step:   5, Red action: DO NOTHING, Blue reward:0.95
step:   6, Red action: DO NOTHING, Blue reward:0.95
step:   7, Red action: DO NOTHING, Blue reward:0.95
step:   8, Red action: DO NOTHING, Blue reward:0.95
step:   9, Red action: DO NOTHING, Blue reward:0.95
step:  10, Red action: DO NOTHING, Blue reward:0.95
step:  11, Red action: DO NOTHING, Blue reward:0.95
step:  12, Red action: DO NOTHING, Blue reward:0.95
step:  13, Red action: DO NOTHING, Blue reward:0.95
step:  14, Red action: DO NOTHING, Blue reward:0.95
step:  15, Red action: DO NOTHING, Blue reward:0.95
step:  16, Red action: DO NOTHING, Blue reward:0.95
step:  17, Red action: DO NOTHING, Blue reward:0.95
step:  18, Red action: DO NOTHING, Blue reward:0.95
step:  19, R

step:  29, Red action: DO NOTHING, Blue reward:-0.80
step:  30, Red action: DO NOTHING, Blue reward:-0.80
step:  31, Red action: DO NOTHING, Blue reward:-0.80
step:  32, Red action: DO NOTHING, Blue reward:-0.80
step:  33, Red action: DO NOTHING, Blue reward:-0.80
step:  34, Red action: DO NOTHING, Blue reward:-0.80


Since the agent does nothing most of the time, let's only print the steps where it performs an attack.

In [7]:
env.reset()
for step in range(100):
    step_num = env.game.step_counter
    obs, reward, terminated, truncated, info = env.step(0)
    red = friendly_output_red_action(info)
    if red.startswith("ATTACK"):
        print(f"step: {step_num:3}, Red action: {friendly_output_red_action(info)}, Blue reward:{reward:.2f}" )

2025-03-21 11:07:50,304: Resetting environment, episode 1, avg. reward: 17.049999999999983


2025-03-21 11:07:50,306: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_1.json


step:  25, Red action: ATTACK from client_1, Blue reward:0.20
step:  49, Red action: ATTACK from client_1, Blue reward:-0.80


step:  70, Red action: ATTACK from client_1, Blue reward:-0.80
step:  95, Red action: ATTACK from client_1, Blue reward:-0.80


## Red Configuration

There are two important parts of the YAML config for varying red agent behaviour.

### Red agent settings
Here is an annotated config for the red agent in the data manipulation scenario.

```yaml
  - ref: data_manipulation_attacker  # name of agent
    team: RED # not used, just for human reference
    type: red-database-corrupting-agent  # type of agent - this lets primaite know which agent class to use

    # These actions are passed to the RedDatabaseCorruptingAgent init method, they dictate the schedule of attacks
    agent_settings:
      possible_start_nodes: [client_1, client_2]  # List of clients the attack can start from
      target_application: data-manipulation-bot
      start_step: 25  # first attack at step 25
      frequency: 20  # attacks will happen every 20 steps (on average)
      variance: 5  # the timing of attacks will vary by up to 5 steps earlier or later
```

### Malicious application settings
The red agent uses an application called `DataManipulationBot` which leverages a node's `DatabaseClient` to send a malicious SQL query to the database server. Here's an annotated example of how this is configured in the yaml *(with impertinent config items omitted)*:

```yaml
simulation:
  network:
    nodes:
      - hostname: client_1
        type: computer
        ip_address: 192.168.10.21
        subnet_mask: 255.255.255.0
        default_gateway: 192.168.10.1
        applications:
          - type: data-manipulation-bot
          options:
            port_scan_p_of_success: 0.8 # Probability that port scan is successful
            data_manipulation_p_of_success: 0.8 # Probability that SQL attack is successful
            payload: "DELETE" # The SQL query which causes the attack (this has to be DELETE)
            server_ip: 192.168.1.14 # IP address of server hosting the database
          - type: database-client # Database client must be installed in order for DataManipulationBot to function
          options:
            db_server_ip: 192.168.1.14 # IP address of server hosting the database
```

## Editing red agent settings

### Removing randomness from attack timing

We can make the attacks happen at completely predictable intervals if we edit the red agent's settings to set variance to 0.

In [8]:
change = yaml.safe_load("""
  possible_start_nodes: [client_1]
  target_application: DataManipulationBot
  start_step: 25
  frequency: 20
  variance: 0
""")

with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)
    for agent in cfg['agents']:
      if agent['ref'] == "data_manipulation_attacker":
        print(f"{agent['agent_settings']=}")
        agent['agent_settings'] = change
        print(f"{agent['agent_settings']=}")

env = PrimaiteGymEnv(env_config = cfg)
env.reset()
for step in range(100):
    step_num = env.game.step_counter
    obs, reward, terminated, truncated, info = env.step(0)
    red = friendly_output_red_action(info)
    if red.startswith("ATTACK"):
        print(f"step: {step_num:3}, Red action: {friendly_output_red_action(info)}, Blue reward:{reward:.2f}" )

agent['agent_settings']={'possible_start_nodes': ['client_1', 'client_2'], 'target_application': 'data-manipulation-bot', 'start_step': 25, 'frequency': 20, 'variance': 5}
agent['agent_settings']={'possible_start_nodes': ['client_1'], 'target_application': 'DataManipulationBot', 'start_step': 25, 'frequency': 20, 'variance': 0}


2025-03-21 11:07:51,094: PrimaiteGymEnv RNG seed = None


2025-03-21 11:07:51,095: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:07:51,096: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


step:  25, Red action: ATTACK from client_1, Blue reward:1.00
step:  45, Red action: ATTACK from client_1, Blue reward:1.00


step:  65, Red action: ATTACK from client_1, Blue reward:1.00
step:  85, Red action: ATTACK from client_1, Blue reward:1.00


### Making the start node always the same

Normally, the agent randomly chooses between the nodes in its action space to send attacks from:

In [9]:
# Open the config without changing anything
with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)

env = PrimaiteGymEnv(env_config = cfg)
env.reset()
for ep in range(12):
    env.reset()
    for step in range(31):
        step_num = env.game.step_counter
        obs, reward, terminated, truncated, info = env.step(0)
        red = friendly_output_red_action(info)
        if red.startswith("ATTACK"):
            print(f"Episode: {ep:2}, step: {step_num:3}, Red action: {friendly_output_red_action(info)}" )

2025-03-21 11:07:52,322: PrimaiteGymEnv RNG seed = None


2025-03-21 11:07:52,323: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:07:52,324: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


2025-03-21 11:07:52,373: Resetting environment, episode 1, avg. reward: 0.0


2025-03-21 11:07:52,374: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_1.json


...

Episode:  9, step:  25, Red action: ATTACK from client_2


2025-03-21 11:07:56,992: Resetting environment, episode 12, avg. reward: 20.449999999999996


2025-03-21 11:07:56,993: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_12.json


Episode: 10, step:  25, Red action: ATTACK from client_2


Episode: 11, step:  25, Red action: ATTACK from client_2


We can make the agent always start on a node of our choice letting that be the only node in the agent's action space.

In [10]:
change = yaml.safe_load("""
    agent_settings:
      possible_start_nodes: [client_1]
      target_application: DataManipulationBot

    action_space:
      action_map:
        0:
          action: do-nothing
          options: {}
        1:
          action: node-application-execute
          options:
            node_name: client_1
            application_name: DataManipulationBot
""")

with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)
    for agent in cfg['agents']:
      if agent['ref'] == "data_manipulation_attacker":
        agent.update(change)

env = PrimaiteGymEnv(env_config = cfg)
env.reset()
for ep in range(12):
    env.reset()
    for step in range(31):
        step_num = env.game.step_counter
        obs, reward, terminated, truncated, info = env.step(0)
        red = friendly_output_red_action(info)
        if red.startswith("ATTACK"):
            print(f"Episode: {ep:2}, step: {step_num:3}, Red action: {friendly_output_red_action(info)}" )

2025-03-21 11:07:57,501: PrimaiteGymEnv RNG seed = None


2025-03-21 11:07:57,503: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:07:57,504: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


2025-03-21 11:07:57,551: Resetting environment, episode 1, avg. reward: 0.0


2025-03-21 11:07:57,552: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_1.json


...

Episode: 10, step:   5, Red action: ATTACK from client_1
Episode: 10, step:  10, Red action: ATTACK from client_1
Episode: 10, step:  15, Red action: ATTACK from client_1
Episode: 10, step:  20, Red action: ATTACK from client_1
Episode: 10, step:  25, Red action: ATTACK from client_1


2025-03-21 11:08:02,236: Resetting environment, episode 12, avg. reward: 29.799999999999997


2025-03-21 11:08:02,237: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_12.json


Episode: 10, step:  30, Red action: ATTACK from client_1
Episode: 11, step:   5, Red action: ATTACK from client_1


Episode: 11, step:  10, Red action: ATTACK from client_1
Episode: 11, step:  15, Red action: ATTACK from client_1
Episode: 11, step:  20, Red action: ATTACK from client_1
Episode: 11, step:  25, Red action: ATTACK from client_1
Episode: 11, step:  30, Red action: ATTACK from client_1


### Make the attack less likely to succeed.

We can change the success probabilities within the data manipulation bot application. When the attack succeeds, the reward goes down.

Setting the probabilities to 1.0 means the attack always succeeds - the reward will always drop

Setting the probabilities to 0.0 means the attack always fails - the reward will never drop.

In [11]:
# Make attack always succeed.
change = yaml.safe_load("""
      applications:
      - ref: data_manipulation_bot
        type: data-manipulation-bot
        options:
          port_scan_p_of_success: 1.0
          data_manipulation_p_of_success: 1.0
          payload: "DELETE"
          server_ip: 192.168.1.14
      - ref: client_1_web_browser
        type: web-browser
        options:
          target_url: http://arcd.com/users/
      - ref: client_1_database_client
        type: database-client
        options:
          db_server_ip: 192.168.1.14
""")

with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)
    cfg['simulation']['network']
    for node in cfg['simulation']['network']['nodes']:
      if node['hostname'] in ['client_1', 'client_2']:
        node['applications'] = change['applications']

env = PrimaiteGymEnv(env_config = cfg)
env.reset()
for ep in range(5):
    env.reset()
    for step in range(36):
        step_num = env.game.step_counter
        obs, reward, terminated, truncated, info = env.step(0)
        red = friendly_output_red_action(info)
        if step_num == 35:
            print(f"Episode: {ep:2}, step: {step_num:3}, Reward: {reward:.2f}" )

2025-03-21 11:08:02,939: PrimaiteGymEnv RNG seed = None


2025-03-21 11:08:02,940: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:08:02,941: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


2025-03-21 11:08:02,991: Resetting environment, episode 1, avg. reward: 0.0


2025-03-21 11:08:02,992: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_1.json


...

Episode:  2, step:  35, Reward: -0.80


2025-03-21 11:08:04,858: Resetting environment, episode 5, avg. reward: 17.09999999999998


2025-03-21 11:08:04,859: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_5.json


Episode:  3, step:  35, Reward: -0.80


Episode:  4, step:  35, Reward: -0.80


In [12]:
# Make attack always fail.
change = yaml.safe_load("""
      applications:
      - ref: data_manipulation_bot
        type: data-manipulation-bot
        options:
          port_scan_p_of_success: 0.0
          data_manipulation_p_of_success: 0.0
          payload: "DELETE"
          server_ip: 192.168.1.14
      - ref: client_1_web_browser
        type: web-browser
        options:
          target_url: http://arcd.com/users/
      - ref: client_1_database_client
        type: database-client
        options:
          db_server_ip: 192.168.1.14
""")

with open(data_manipulation_config_path(), 'r') as f:
    cfg = yaml.safe_load(f)
    cfg['simulation']['network']
    for node in cfg['simulation']['network']['nodes']:
      if node['hostname'] in ['client_1', 'client_2']:
        node['applications'] = change['applications']

env = PrimaiteGymEnv(env_config = cfg)
env.reset()
for ep in range(5):
    env.reset()
    for step in range(36):
        step_num = env.game.step_counter
        obs, reward, terminated, truncated, info = env.step(0)
        red = friendly_output_red_action(info)
        if step_num == 35:
            print(f"Episode: {ep:2}, step: {step_num:3}, Reward: {reward:.2f}" )

2025-03-21 11:08:05,411: PrimaiteGymEnv RNG seed = None


2025-03-21 11:08:05,412: Resetting environment, episode 0, avg. reward: 0.0


2025-03-21 11:08:05,413: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_0.json


2025-03-21 11:08:05,463: Resetting environment, episode 1, avg. reward: 0.0


2025-03-21 11:08:05,463: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_1.json


...

Episode:  2, step:  35, Reward: 1.00


2025-03-21 11:08:07,355: Resetting environment, episode 5, avg. reward: 34.099999999999994


2025-03-21 11:08:07,356: Saving agent action log to /home/runner/primaite/4.0.0/sessions/2025-03-21/11-07-46/agent_actions/episode_5.json


Episode:  3, step:  35, Reward: 1.00


Episode:  4, step:  35, Reward: 1.00
